In [13]:
import os
import pandas as pd
import itertools
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows

In [14]:

def calculate_proportions(on_off_df, neuron_types):
    proportions = {}
    for neuron_type in neuron_types:
        neuron_count = len(on_off_df[on_off_df[neuron_type] == 1])
        proportions[neuron_type] = neuron_count
        proportions[f"{neuron_type}_Proportion"] = neuron_count / len(on_off_df.index)
    return proportions

def calculate_intra_overlap(on_off_df, neuron_pairs, proportions):
    overlap_results = {}
    for pair in neuron_pairs:
        neuron_a, neuron_b = pair
        overlap = len(on_off_df[(on_off_df[neuron_a] == 1) & (on_off_df[neuron_b] == 1)])
        total = proportions[neuron_a] + proportions[neuron_b]
        if total > 0:
            overlap_results[f"{neuron_a}_{neuron_b}_O"] = 2 * overlap / total
            overlap_results[f"{neuron_a}_{neuron_b}_P"] = 2 * proportions[neuron_a] * proportions[neuron_b] / (len(on_off_df.index) * total)
        else:
            overlap_results[f"{neuron_a}_{neuron_b}_O"] = 0
            overlap_results[f"{neuron_a}_{neuron_b}_P"] = 0
    return overlap_results

def calculate_cross_type_overlap(df1, df2, neuron_pairs):
    overlap_results = {}
    
    for type in neuron_pairs:
        type1, type2 = type
        neurons_df1 = set(df1[df1[type1] == 1].index)
        neurons_df2 = set(df2[df2[type2] == 1].index)
        
        overlap_count = len(neurons_df1 & neurons_df2)
        total = (len(neurons_df1) + len(neurons_df2))
        jaccard = 2 * overlap_count / total if (len(neurons_df1) + len(neurons_df2)) > 0 else 0
        overlap_ratio1 = overlap_count / len(neurons_df1) if len(neurons_df1) > 0 else 0
        overlap_ratio2 = overlap_count / len(neurons_df2) if len(neurons_df2) > 0 else 0
        dice_coeff = 2 * len(neurons_df1) * len(neurons_df2) / (len(df1) * total) if (len(neurons_df1) + len(neurons_df2)) > 0 else 0
        
        key_prefix = f"{type1}_to_{type2}"
        overlap_results[f"{key_prefix}_overlap_count"] = overlap_count
        overlap_results[f"{key_prefix}_overlap_observed"] = jaccard
        overlap_results[f"{key_prefix}_overlap_ratio1"] = overlap_ratio1
        overlap_results[f"{key_prefix}_overlap_ratio2"] = overlap_ratio2
        overlap_results[f"{key_prefix}_overlap_predict"] = dice_coeff
    
    return overlap_results

In [15]:


root_path = 'F:/AAA-RXC/Results'
mouse_list = ['PL1-4sessions','PL2-4sessions','PL4-4sessions','PL5-4sessions','PL6-4sessions','PL7-4sessions','PL8-4sessions']
session_files = ['session_3_ONOFF.xlsx', 'session_2_ONOFF.xlsx']  #####注意数据来源！！！！
neuron_types = ['cs_ON', 'freezing_ON','sniff_ON', 'sniffed_ON']
neuron_pairs = [('sniffed_ON', 'freezing_ON'), ('sniffed_ON', 'cs_ON')]

output_excel_path = os.path.join(root_path, 'all_mice_overlap_analysis1.xlsx')
with pd.ExcelWriter(output_excel_path, engine='openpyxl') as writer:
    all_intra_results = []
    all_cross_type_results = []
    
    for mouse_name in mouse_list:
        print(f'正在处理 {mouse_name}')
        mouse_path = os.path.join(root_path, mouse_name)
        
        session_dfs = {}
        for session_file in session_files:
            file_path = os.path.join(mouse_path, session_file)
            df = pd.read_excel(file_path)
            df.columns = ['Neurons'] + df.columns[1:].tolist()
            df = df.set_index('Neurons')
            session_dfs[session_file] = df
        
        intra_results = []
        for session_name, df in session_dfs.items():
            proportions = calculate_proportions(df, neuron_types)
            overlap_results = calculate_intra_overlap(df, neuron_pairs, proportions)
            
            result = {
                'mouse': mouse_name,
                'session': session_name,
                'all_neuron': len(df.index)
            }
            result.update(proportions)
            result.update(overlap_results)
            intra_results.append(result)
        
        for result in intra_results:
            all_intra_results.append(result)
        
        # intra_df = pd.DataFrame(intra_results)
        # sheet_name_intra = f"{mouse_name}_intra"
        # if len(sheet_name_intra) > 31:
        #     sheet_name_intra = sheet_name_intra[:31]
        # intra_df.to_excel(writer, sheet_name=sheet_name_intra, index=False)
        
        cross_type_results = []
        df1 = session_dfs['session_3_ONOFF.xlsx']   #####注意数据来源！！！！
        df2 = session_dfs['session_2_ONOFF.xlsx']   #####注意数据来源！！！！
        neuron_pairs_across = [('sniffed_ON', 'freezing_ON'), ('sniffed_ON', 'cs_ON'), ]
        
        overlap_results = calculate_cross_type_overlap(df1, df2, neuron_pairs_across)
            
        result = {
            'mouse': mouse_name,
            'session_pair': f"{session_files[0]} vs {session_files[1]}",
            'session1': session_files[0],
            'session2': session_files[1],
            'total_neurons_session1': len(df1.index),
            'total_neurons_session2': len(df2.index)
        }
        result.update(overlap_results)
        cross_type_results.append(result)
        
        for result in cross_type_results:
            all_cross_type_results.append(result)
        
        # cross_type_df = pd.DataFrame(cross_type_results)
        # sheet_name_cross = f"{mouse_name}_cross"
        # if len(sheet_name_cross) > 31:
        #     sheet_name_cross = sheet_name_cross[:31]
        # cross_type_df.to_excel(writer, sheet_name=sheet_name_cross, index=False)
    
    if all_intra_results:
        all_intra_df = pd.DataFrame(all_intra_results)
        all_intra_df.to_excel(writer, sheet_name='All_Mice_Intra_Summary', index=False)
    
    if all_cross_type_results:
        all_cross_type_df = pd.DataFrame(all_cross_type_results)
        all_cross_type_df.to_excel(writer, sheet_name='All_Mice_Cross_Summary', index=False)

print("所有分析完成！")
print(f"数据保存在：{output_excel_path}")

正在处理 PL1-4sessions
正在处理 PL2-4sessions
正在处理 PL4-4sessions
正在处理 PL5-4sessions
正在处理 PL6-4sessions
正在处理 PL7-4sessions
正在处理 PL8-4sessions
所有分析完成！
数据保存在：F:/AAA-RXC/Results\all_mice_overlap_analysis1.xlsx
